# Phase P1: Setup, Hardware Preflight & Dataset Preparation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/00_setup_preflight.ipynb)


In [ ]:
import os, sys, json, subprocess
# Check if running in smoke/CI mode
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/revllm/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/runs', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/probes', exist_ok=True)
print(f"Execution mode: {'Smoke (CPU/CI)' if SMOKE else 'Colab GPU'}")


In [ ]:
# Hardware and environment verification
import torch

env_data = {
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "None",
    "python_version": sys.version.split()[0],
}

if not torch.cuda.is_available() and not SMOKE:
    raise RuntimeError("No GPU detected! Ensure Colab runtime is set to GPU (T4).")

os.makedirs("results", exist_ok=True)
with open("results/env.json", "w") as f:
    json.dump(env_data, f, indent=2)
print("Hardware environment recorded:", json.dumps(env_data, indent=2))


In [ ]:
# Install package and run preflight verification
cmd = [sys.executable, "-m", "pip", "install", "-e", "."]
if not SMOKE and IN_COLAB:
    subprocess.run(cmd, check=True)

# Run preflight verification script
from scripts.preflight import run_preflight
preflight_results = run_preflight()
print("Preflight verification status:", preflight_results.get("arch_checks"))


In [ ]:
# Prepare dataset and tokenizer
from scripts.build_data import main as build_data_main
# In smoke mode, creates tiny dataset; in full Colab mode builds full FineWeb-Edu split
build_data_main()
if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/data_manifest.json", "/content/drive/MyDrive/revllm/data/data_manifest.json")
    if os.path.exists("data/tokenizer.json"):
        shutil.copy2("data/tokenizer.json", "/content/drive/MyDrive/revllm/data/tokenizer.json")
print("Dataset preparation complete.")
